[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C64_ML_Knowledge_QA_Course/05_rapid_fire/05_rapid_fire_bank.ipynb)

# 05 · 快问快答题库与自测（150+ 题结构化题库 / 闪卡引擎 / SM-2 间隔重复 / 弱项报告 / 90 分钟复习排程）

目标：把「面试前反复自测」从一份静态的题目清单，变成**一台会记住你哪里薄弱、并自动安排复习时间的引擎**。

本 notebook 你会亲手实现：
1. **环境自检**
2. **150+ 题结构化题库** —— 按 ml_basics / optimization / architectures / eval_stats / cv_entry 五大主题内置
3. **SM-2 间隔重复算法** —— 从零实现易记因子(EF)/复习间隔/复习次数的更新公式
4. **闪卡引擎** —— 随机抽题、隐藏答案、按自评分调度下一次复习
5. **弱项报告生成器** —— 按主题聚合掌握度，找出你最该花时间的地方
6. **90 分钟复习清单自动排程** —— 按弱项权重把 90 分钟分配到各主题，总和严格等于 90

> 心智模型：**面试前复习的边际收益，来自把时间导向"刚好快忘、最该复习"的那些题，而不是从头到尾重读。**

## 0 · 环境自检

本课全程只用标准库 + numpy。没有 GPU 依赖、不联网、不下载数据。

In [ ]:
import sys, math
import numpy as np
from collections import defaultdict

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'

rng = np.random.default_rng(11)
print('\n✅ 环境自检通过：本模块不需要 GPU、不需要联网，全程固定 seed 保证可复现。')

## 1 · 题库数据结构

每张卡片是一个字典：`topic`（主题）/ `tier`（must/bonus/boundary）/ `freq`（high/mid/low）/
`q`（题目）/ `a`（一句话答案）/ `follow`（典型追问）/ `pitfall`（踩雷点）。
用一个小 helper `QA(...)` 减少重复代码。

In [ ]:
_next_id = [0]

def QA(topic, tier, freq, q, a, follow, pitfall):
    """tier ∈ {'must','bonus','boundary'}；freq ∈ {'high','mid','low'}"""
    _next_id[0] += 1
    return {'id': f'{topic}_{_next_id[0]:03d}', 'topic': topic, 'tier': tier, 'freq': freq,
            'q': q, 'a': a, 'follow': follow, 'pitfall': pitfall}

assert QA('x', 'must', 'high', 'Q?', 'A.', 'F?', 'P.')['tier'] == 'must'
print('QA() 就位：每张卡片是 topic/tier/freq/q/a/follow/pitfall 七个字段。')

## 2 · 五大主题题库（结构化数据）

分主题内置，每题都是本课程模块 01–04（+ CV 专项入口）的对应内容压缩成的一句话问答。
完整讲解见对应模块，这里只做**可自测的最小单元**。

In [ ]:
ML_BASICS = [
    QA('ml_basics','must','high','高方差的信号是什么？','训练误差低、验证误差高，且差距随复杂度拉大。','怎么和高偏差区分？','把"验证误差高"单独当高方差信号，忘了对比训练误差。'),
    QA('ml_basics','must','high','高偏差的信号是什么？','训练误差和验证误差都高，且都远高于可接受水平。','怎么修？','以为加更多数据能解决高偏差——高偏差要靠加复杂度或加特征。'),
    QA('ml_basics','bonus','mid','双下降现象是什么？','模型复杂度越过插值阈值后，测试误差先升后再次下降。','经典偏差-方差理论为什么解释不了？','以为双下降否定了偏差-方差权衡，它只是补充了理论没覆盖的区间。'),
    QA('ml_basics','must','high','L1为什么产生稀疏解？','L1等高线是菱形，最优点容易落在坐标轴顶点（系数恰好为0）。','L2呢？','混淆"权重变小"和"权重变0"，后者才是稀疏性定义。'),
    QA('ml_basics','must','high','L2为什么不产生稀疏解？','L2等高线是圆形，最优点几乎不会恰好落在坐标轴上。','那L2解决了什么问题？','抑制权重整体幅度，但不做特征选择。'),
    QA('ml_basics','bonus','mid','dropout为什么等价于隐式集成？','每次前向相当于采样一个子网络，等价于对指数级子网络平均。','推理时为什么要缩放？','保证期望输出和训练时一致，忘了缩放会导致输出尺度不一致。'),
    QA('ml_basics','bonus','mid','early stopping和L2有什么等价性？','在线性模型/简单假设下，两者都限制了参数能到达的空间大小。','等价性在什么条件下成立？','把这个等价性当成普遍成立的定理，忽略了它的前提假设。'),
    QA('ml_basics','must','mid','数据增强为什么算一种正则化？','人为扩大训练分布覆盖范围，限制模型对训练集specific噪声的拟合。','增强太强会怎样？','小模型+强增强反而欠拟合。'),
    QA('ml_basics','must','high','k折交叉验证的作用是什么？','用有限数据估计模型在未见数据上的泛化误差，降低单次划分的方差。','k怎么选？','k太小方差大，k太大（如LOO）计算贵且估计有偏。'),
    QA('ml_basics','bonus','mid','嵌套交叉验证解决什么问题？','超参搜索本身在验证集上会过拟合，嵌套CV把超参选择也纳入外层验证。','为什么普通CV选超参会乐观偏差？','在同一份验证集上反复选超参，等价于多重比较。'),
    QA('ml_basics','must','high','为什么时序数据不能用普通k折？','会让未来样本进入训练集预测过去，造成信息泄漏。','应该怎么做？','按时间顺序切分，只能用过去预测未来。'),
    QA('ml_basics','bonus','mid','为什么同一病人多条记录要用group k-fold？','同一分组内的样本天然相关，混在训练/验证两侧会造成泄漏。','怎么识别这类分组？','忽略了"同一实体产生多条记录"这种隐式分组结构。'),
    QA('ml_basics','bonus','low','留一法(LOO-CV)的优缺点？','偏差最小（几乎用全部数据训练）但方差大、计算量随n线性增长。','什么时候适合用？','小样本、计算成本可接受时。'),
    QA('ml_basics','must','high','类别不平衡时准确率为什么是坏指标？','全猜多数类也能有很高准确率，掩盖了稀有类完全学不到的事实。','该看什么？','稀有类的recall/PR-AUC，见C64-04。'),
    QA('ml_basics','bonus','mid','类别不平衡处理谱系有哪几大类？','重采样、重加权（含Focal/CB Loss）、解耦训练三大类。','检测里有什么特有做法？','repeat factor sampling等，见C58-01。'),
    QA('ml_basics','must','high','bagging为什么降方差？','并行训练多个高方差基学习器再平均，随机误差相互抵消。','对偏差有帮助吗？','基本没有，如果基学习器偏差大，平均再多个也没用。'),
    QA('ml_basics','must','high','boosting为什么降偏差？','顺序训练，每一轮纠正前一轮的残差，逐步逼近真值。','boosting容易过拟合吗？','是，需要正则化（收缩率、树深度限制等）。'),
    QA('ml_basics','bonus','mid','随机森林为什么对超参数不敏感？','平均本身自带正则化，单棵树的过拟合被大量平均抵消。','GBDT为什么敏感？','顺序拟合残差，学习率/树数/树深任一没调好都会显著影响结果。'),
    QA('ml_basics','bonus','low','stacking和bagging/boosting的本质区别？','stacking用一个元学习器学习"怎么组合"基学习器的输出，而不是简单平均或顺序修正。','元学习器怎么避免过拟合？','用交叉验证的out-of-fold预测训练元学习器。'),
    QA('ml_basics','bonus','mid','生成式和判别式模型的区别？','生成式建模联合分布P(x,y)能采样，判别式直接建模P(y|x)只能分类。','小样本场景为什么常选生成式？','对分布做了更强假设，方差更低。'),
    QA('ml_basics','bonus','mid','维度灾难是什么？','高维空间样本变稀疏，任意两点距离趋于相等，基于距离的方法失效。','流形假设怎么缓解？','真实数据集中在低维流形上，有效维度远低于原始维度。'),
    QA('ml_basics','must','mid','特征选择和特征工程的区别？','特征工程是构造新特征，特征选择是从已有特征里挑一个子集。','什么时候特征越多越差？','过拟合、多重共线性、推理成本上升时。'),
    QA('ml_basics','must','mid','判别阈值和决策边界的关系？','阈值决定了决策边界在概率空间的具体位置，阈值变化即边界平移。','阈值该怎么选？','见C64-04代价敏感阈值选择。'),
    QA('ml_basics','bonus','low','为什么树模型不需要归一化特征？','基于阈值切分不依赖特征的绝对尺度，只依赖相对顺序。','线性模型/神经网络为什么需要？','梯度下降对特征尺度敏感，尺度不一会导致收敛困难。'),
    QA('ml_basics','bonus','mid','什么时候增加数据比增加模型容量更有效？','当误差主要来自方差（高方差信号）而非偏差时。','高偏差时加数据有用吗？','基本没用，需要加复杂度或改特征。'),
    QA('ml_basics','boundary','low','为什么"更多数据"不能永远解决偏差问题？','偏差来自模型假设空间本身的局限，数据量增大不会扩大假设空间。','有没有例外？','某些非参数模型理论上假设空间会随数据增长，但工程上很少见。'),
    QA('ml_basics','bonus','low','Occam剃刀原则在模型选择里怎么用？','同等验证表现下，优先选更简单的模型，泛化风险更低、可维护性更好。','这和正则化的关系？','正则化本质上是把Occam剃刀显式编码进了损失函数。'),
    QA('ml_basics','must','mid','什么是标签泄漏(label leakage)？','特征里包含了只有在预测目标已知之后才能获得的信息。','怎么发现？','某个特征单独就能预测出接近完美的结果时要警惕。'),
    QA('ml_basics','bonus','mid','Bootstrap采样在bagging里的具体机制？','有放回抽样，每个基学习器的训练集期望覆盖约63.2%的原始样本。','剩下36.8%有什么用？','可以做out-of-bag估计，不需要单独留出验证集。'),
    QA('ml_basics','bonus','low','为什么集成的基学习器需要"好而不同"？','完全相同的基学习器平均后方差不会降低；完全无关但很差的基学习器平均后整体也差。','怎么衡量"不同"？','基学习器预测误差的相关性，相关性越低集成收益越大。'),
    QA('ml_basics','must','mid','什么时候不该用交叉验证？','时序数据（未来泄漏）、分组数据（跨组泄漏）、以及数据量极大计算成本过高时。','那该用什么？','时间顺序切分/分组切分/单次留出验证集。'),
    QA('ml_basics','bonus','mid','为什么dropout在卷积层里效果不如全连接层明显？','卷积层的空间相邻激活值高度相关，随机丢弃单个像素信息很容易从邻居恢复。','有什么替代？','SpatialDropout/DropBlock，成块丢弃。'),
    QA('ml_basics','bonus','low','正则化路径(regularization path)是什么？','正则化强度从0到大变化时，模型系数变化的完整轨迹。','L1和L2的路径形状有什么不同？','L1路径是分段线性且会有系数精确归零的拐点，L2路径是平滑收缩。'),
    QA('ml_basics','boundary','low','偏差-方差分解在深度学习里是否还完全适用？','经典分解假设的是固定容量的模型族，深度网络的隐式正则化让分解不再是简单可加的。','那还有什么框架能用？','双下降相关的理论工作在尝试补充，但还没有统一框架，见本模块§9前沿讨论。'),
    QA('ml_basics','must','mid','什么时候模型选择该用单次留出验证集而不是交叉验证？','数据量足够大时单次留出的方差已经足够小，交叉验证的额外计算成本不划算。','数据量小怎么办？','用k折或嵌套k折降低验证方差，见本节前面的CV相关题。'),
    QA('ml_basics','bonus','mid','半监督学习的核心假设是什么？','未标注数据的分布结构（聚类/流形）能帮助改善决策边界，前提是这个结构假设成立。','什么时候半监督会帮倒忙？','聚类结构和真实类别边界不一致时，反而会引入噪声。'),
    QA('ml_basics','bonus','low','主动学习和不确定性采样的关系？','主动学习选择"模型最不确定"的样本优先标注，以最小标注成本换取最大信息增益。','这和检测数据闭环的触发策略是什么关系？','本质相同的思想被用在感知数据挖掘场景，见C58-03。'),
]

assert len(ML_BASICS) >= 30, len(ML_BASICS)
print(f'ml_basics 题库：{len(ML_BASICS)} 题')

In [ ]:
OPTIMIZATION = [
    QA('optimization','must','high','SGD和批量GD的区别？','SGD每次用单个/小批样本估计梯度，批量GD用全部数据；SGD噪声大但每步快得多。','为什么SGD的噪声反而有用？','噪声能帮助跳出尖锐极小值，见泛化相关追问。'),
    QA('optimization','must','high','Momentum解决什么问题？','累积历史梯度方向，抑制震荡、加速沿一致方向下降，帮助穿过平坦区域。','和Nesterov的区别？','Nesterov先按动量方向"预看一步"再计算梯度，修正更超前。'),
    QA('optimization','bonus','mid','AdaGrad的致命缺陷是什么？','历史梯度平方累积单调增长，导致学习率单调递减趋于0，后期几乎不更新。','RMSProp怎么修复？','用指数移动平均替代累积和，让"历史"会遗忘。'),
    QA('optimization','must','high','Adam结合了哪两种机制？','一阶矩（动量）和二阶矩（自适应学习率，类似RMSProp）的估计。','Adam的两个偏差修正项是干什么的？','初始时刻矩估计有偏向0的偏差，修正项用于早期步数校正。'),
    QA('optimization','bonus','high','为什么CV检测任务里SGD+Momentum常常仍是首选？','Adam收敛快但常收敛到更尖锐的极小值，泛化性有时不如SGD+Momentum找到的平坦极小值。','什么时候该用Adam？','Transformer、小数据、快速实验迭代阶段。'),
    QA('optimization','must','high','AdamW和Adam+L2为什么不等价？','Adam的自适应缩放会重新缩放梯度含L2项在内，AdamW把权重衰减解耦到梯度更新之外直接作用于参数。','解耦后有什么好处？','权重衰减强度不再受自适应学习率干扰，更符合直觉。'),
    QA('optimization','must','mid','step decay和cosine decay的区别？','step在固定epoch阶梯式下降；cosine按余弦曲线平滑衰减到0，末期收尾更平滑。','one-cycle policy是什么？','学习率先升后降的单周期调度，通常配合动量反向调度。'),
    QA('optimization','must','high','Warmup为什么必要？','训练初期梯度方差大、自适应二阶矩估计不准，直接用目标学习率容易不稳定甚至发散。','Transformer为什么对warmup格外敏感？','LN+自注意力初期梯度尺度波动更大。'),
    QA('optimization','must','high','梯度消失的成因是什么？','链式法则里多个小于1的导数连乘，深层网络里梯度指数级衰减。','怎么缓解？','残差连接、归一化层、合适的初始化。'),
    QA('optimization','must','mid','梯度爆炸怎么应对？','梯度裁剪(gradient clipping)——按范数或按值截断梯度。','裁剪会不会影响收敛方向？','按范数裁剪只缩放大小不改变方向，通常安全。'),
    QA('optimization','bonus','mid','残差连接为什么缓解梯度消失？','恒等映射提供了一条梯度可以直接跨层传播的短路路径，不必完全依赖连乘的权重路径。','这和集成视角有什么关系？','残差网络也可以看成大量不同深度子路径的隐式集成。'),
    QA('optimization','must','mid','Xavier初始化的核心直觉？','让前向传播的激活方差和反向传播的梯度方差在层间保持一致，避免逐层放大或衰减。','对什么激活函数适用？','假设激活关于0对称，如tanh/sigmoid。'),
    QA('optimization','must','mid','He初始化为什么专为ReLU设计？','ReLU让一半激活变0方差减半，He初始化把方差乘2做补偿。','用错初始化会怎样？','激活方差逐层衰减或爆炸，训练初期就可能失败。'),
    QA('optimization','bonus','mid','大batch训练为什么泛化更差？','batch越大梯度噪声越小，更容易收敛到损失曲面尖锐区域，对分布偏移更敏感。','线性缩放规则是什么？','batch放大k倍，学习率也放大k倍（配合warmup）维持有效噪声水平。'),
    QA('optimization','bonus','mid','混合精度训练fp16的溢出问题是什么？','fp16动态范围窄，梯度过小会下溢为0，过大会上溢为inf。','loss scaling怎么解决？','把loss乘一个大常数放大梯度尺度，反传后再除回去，避免下溢。'),
    QA('optimization','bonus','mid','bf16和fp16的差异？','bf16指数位更宽（和fp32相同范围）不易溢出，但尾数位更少精度更低；fp16范围窄但精度相对更高。','什么场景选哪个？','bf16对数值稳定性要求高的大模型训练更友好，fp16对推理加速更常见。'),
    QA('optimization','bonus','low','梯度累积的原理和代价是什么？','多个小batch的梯度累加后再统一更新一次，模拟大batch效果；代价是不省显存的另一半好处（激活值仍按小batch算，速度没有大batch快）。','和真正大batch训练有什么区别？','BN统计量仍然是按小batch算的，除非配合同步BN。'),
    QA('optimization','must','high','MSE和MAE的区别？','MSE对大误差惩罚更重(平方)，对异常值敏感；MAE对所有误差线性惩罚，对异常值更鲁棒但在0点不可导。','怎么选？','有明显异常值倾向MAE或Huber，否则MSE梯度性质更好。'),
    QA('optimization','bonus','mid','Huber Loss解决什么问题？','小误差区域用平方损失（梯度平滑可导），大误差区域切换为线性损失（抑制异常值影响）。','阈值delta怎么选？','按误差分布的典型尺度设定，太小退化成MAE，太大退化成MSE。'),
    QA('optimization','must','high','CE和Focal Loss的区别？','Focal Loss在CE基础上乘一个(1-p)^γ因子，降权已经分类正确的简单样本，聚焦难例。','γ=0时Focal退化成什么？','带α加权的标准交叉熵。'),
    QA('optimization','must','high','Focal Loss的α和γ各自作用？','α处理类别数量不平衡，γ处理难易样本不平衡（降权简单样本）。','两者能同时用吗？','可以，通常同时设置，各自处理不同维度的不平衡。'),
    QA('optimization','bonus','mid','Label Smoothing解决什么问题？','防止模型对训练标签过度自信，把one-hot目标软化，改善校准。','副作用是什么？','轻微损害需要"非常自信输出"的场景，如知识蒸馏的教师模型。'),
    QA('optimization','bonus','low','为什么优化器选择和网络架构有交互？','Transformer的LN+自注意力对二阶矩估计的稳定性更敏感，几乎必须用Adam类优化器；CNN+BN对SGD+Momentum更友好。','有例外吗？','部分工作证明配合合适的warmup和调参，SGD也能训练Transformer，但更难调。'),
    QA('optimization','bonus','mid','什么时候该用学习率warmup+cosine decay的组合？','大batch/大模型训练的标准配置，warmup避免早期不稳定，cosine让末期平滑收敛。','one-cycle和它的区别？','one-cycle学习率先升后降为单峰，通常训练更短总步数下使用。'),
    QA('optimization','boundary','low','为什么Adam有时收敛到差的极小值，完整机制是否有定论？','有多个假设（自适应学习率导致的有效步长偏差、缺乏SGD噪声的隐式正则化）但没有统一定论。','怎么在实践中弥补？','用AdamW+适当权重衰减，或训练后期切换到SGD微调（一种常见工程折中）。'),
    QA('optimization','bonus','mid','为什么梯度裁剪常用在RNN/Transformer里更多？','长序列反向传播容易出现梯度爆炸的极端情况，裁剪是低成本的稳定手段。','裁剪阈值怎么选？','按训练初期梯度范数的分布经验设定，然后监控是否频繁触发裁剪。'),
    QA('optimization','bonus','low','权重衰减和批归一化(scale-invariant层)有什么隐藏交互？','BN让前面层的权重具有尺度不变性，权重衰减此时实际上是在调节有效学习率而非直接限制权重大小。','这意味着什么？','权重衰减在BN网络里的作用机制和无BN网络里不完全一样，调参经验不能直接照搬。'),
    QA('optimization','must','mid','什么时候该用余弦退火重启(warm restart)？','训练容易陷入局部极小值/需要多次探索不同区域时，周期性重新升高学习率跳出当前区域。','和普通cosine decay的区别？','warm restart会周期性重复整个衰减过程，可以配合快照集成使用。'),
    QA('optimization','must','mid','优化器选择上"没有免费午餐"是什么意思？','没有一个优化器/超参数配置在所有任务上都最优，选择必须结合具体损失曲面特性。','被问"哪个优化器最好"该怎么答？','给出场景依赖的判断（架构/数据量/训练预算），而不是给绝对结论。'),
    QA('optimization','bonus','mid','为什么GAN训练比普通监督学习更难调优化器？','生成器和判别器是极小极大博弈，损失曲面非静态（对手在变化），传统收敛理论不完全适用。','有什么常见对策？','谱归一化、双时间尺度更新规则(TTUR)等专门稳定手段。'),
    QA('optimization','bonus','low','学习率过大和过小分别有什么典型症状？','过大：loss震荡甚至发散；过小：收敛极慢，长时间训练loss几乎不降。','怎么快速诊断合适区间？','学习率区间测试(LR range test)，几百步内扫描找到合适范围。'),
]

assert len(OPTIMIZATION) >= 25, len(OPTIMIZATION)
print(f'optimization 题库：{len(OPTIMIZATION)} 题')

In [ ]:
ARCHITECTURES = [
    QA('architectures','must','high','卷积参数量怎么算？','Cin × Cout × k × k（+ Cout个bias），与输入特征图尺寸无关。','FLOPs怎么算？','参数量 × 输出特征图的空间尺寸（H×W）。'),
    QA('architectures','must','high','感受野递推公式是什么？','RF_l = RF_{l-1} + (k_l - 1) × Π(前面所有层的stride)。','有效感受野和理论感受野的区别？','有效感受野往往远小于理论值，中心权重远高于边缘。'),
    QA('architectures','must','high','1×1卷积的三个作用？','跨通道信息融合、升维/降维（控制参数量计算量）、增加非线性。','和全连接层的区别？','1×1卷积逐空间位置独立作用，权重在空间维度共享。'),
    QA('architectures','bonus','mid','深度可分离卷积收益和代价？','参数量计算量大幅下降；代价是通道间交互能力变弱，同参数量下表达能力打折。','什么场景代价可接受？','移动端/车端算力受限场景，用宽度换回部分能力。'),
    QA('architectures','bonus','mid','空洞卷积(dilated conv)的作用和问题？','不增加参数量的情况下扩大感受野；问题是gridding artifact（采样点呈网格状不连续，丢失局部细节）。','怎么缓解gridding artifact？','混合不同膨胀率的空洞卷积（hybrid dilated convolution）。'),
    QA('architectures','bonus','mid','转置卷积棋盘效应从哪来？','kernel size不能被stride整除时，输出像素被重叠覆盖次数不均匀，形成周期性伪影。','怎么避免？','让kernel size是stride整数倍，或用插值+普通卷积替代。'),
    QA('architectures','must','mid','池化和stride卷积的区别？','池化无参数、有一定平移不变性但会丢失信息；stride卷积可学习，保留更多可判别信息但增加参数。','现代架构更倾向哪个？','很多现代检测/分割架构倾向用stride卷积替代池化做下采样。'),
    QA('architectures','must','high','BN训练和推理的不同？','训练用当前mini-batch的均值/方差，推理用训练过程中累积的running mean/var。','为什么这个不同会导致问题？','小batch下训练时的批统计量本身噪声大，和推理时的running统计量差异被放大。'),
    QA('architectures','must','high','BN在检测任务上为什么容易出问题？','检测显存占用大，单卡batch往往很小，BN统计量在小batch下估计噪声很大。','怎么解决？','同步BN（多卡汇总）、GroupNorm（不依赖batch）、冻结BN统计量微调。'),
    QA('architectures','must','high','LN和BN的区别？','LN在样本内（跨特征/通道维）归一化，BN在batch维（跨样本）归一化。','为什么Transformer用LN不用BN？','LN不依赖batch内其他样本，天然适配变长序列和推理batch=1场景。'),
    QA('architectures','bonus','mid','GN(GroupNorm)解决什么问题？','把通道分组后组内归一化，不依赖batch size，弥补小batch下BN的不稳定。','GN和LN的区别？','LN是所有通道一组，GN是通道分成若干组，是两者之间的一般化。'),
    QA('architectures','bonus','low','IN(InstanceNorm)常用在什么场景？','风格迁移等需要去除单样本内部对比度/亮度等风格信息的任务。','IN和BN的区别？','IN对每个样本每个通道单独归一化，不跨样本也不跨通道。'),
    QA('architectures','must','high','残差连接的梯度视角解释？','恒等映射提供梯度可以直接跨层传播的短路路径，避免连乘导致的梯度消失。','残差连接的集成视角？','可以看成大量不同深度子路径的隐式集成。'),
    QA('architectures','must','high','QKV的直觉是什么？','Query表示"在问什么"，Key表示"能被匹配到的索引"，Value表示"匹配到之后取出的内容"。','三者为什么要分开而不用同一个向量？','分开能让"匹配相似度"和"要提取的信息"解耦，表达能力更强。'),
    QA('architectures','must','high','为什么attention要除以√d_k？','点积方差随d_k线性增长，不缩放会让softmax输入尺度过大，梯度趋于饱和。','怎么验证饱和现象？','固定d_k比较缩放前后softmax输出熵，见C64-03对应实验。'),
    QA('architectures','must','mid','多头注意力的作用是什么？','让不同子空间关注不同的关系模式，增加模型表达的多样性。','头数越多越好吗？','不是，头数增加要以每头维度降低为代价，存在权衡。'),
    QA('architectures','bonus','mid','绝对/相对/可学习位置编码的区别？','绝对编码给每个位置固定向量；相对编码建模位置之间的相对偏移；可学习编码让模型自己学出编码值。','外推性问题是什么？','训练长度之外的序列长度上，位置编码的表现可能显著下降。'),
    QA('architectures','bonus','mid','self-attention的O(n²)复杂度来源？','每个位置都要和所有其他位置计算注意力权重，长度为n时是n×n的矩阵运算。','缓解思路有哪些？','稀疏注意力、线性注意力近似、滑动窗口注意力、可变形注意力（见C54-04）。'),
    QA('architectures','must','high','CNN vs Transformer的核心取舍？','CNN局部性+平移不变性是强归纳偏置，小数据下更好训练；Transformer几乎无归纳偏置，需要更多数据/更长训练。','ViT在小数据集上为什么打不过ResNet？','归纳偏置需要靠数据量弥补，小数据下这笔账划不来。'),
    QA('architectures','bonus','mid','参数量和显存的口算方法？','参数量×4字节(fp32)估算模型本身显存，优化器状态(Adam)约再翻2-3倍，加上激活值显存。','为什么优化器状态占的显存经常被忽略？','Adam要存一阶矩+二阶矩，等于模型参数量的2倍额外显存。'),
    QA('architectures','bonus','low','深度和宽度对模型容量的不同影响？','深度增加表达能力的组合复杂度（层级特征），宽度增加单层的表征容量，两者对优化难度的影响也不同。','什么时候优先加深度？','需要更强的层级抽象能力（如更复杂的语义关系）时。'),
    QA('architectures','bonus','mid','为什么现代架构偏爱堆叠3×3小卷积而不是大卷积核？','相同感受野下参数量更少（VGG的经验），且多层非线性堆叠表达能力更强。','那为什么最近又流行5×5甚至7×7大核？','大核在保持较小参数增量下能显著扩大有效感受野，配合深度可分离卷积代价可控（见C53-03 RTMDet）。'),
    QA('architectures','bonus','low','Batch Norm和权重衰减有什么隐藏交互？','BN让前面层权重具有尺度不变性，权重衰减实际上在调节有效学习率而非直接限制权重大小。','这对调参有什么影响？','权重衰减的作用机制和无BN网络不一样，经验不能直接照搬。'),
    QA('architectures','must','mid','skip connection和U-Net的skip connection有什么区别？','ResNet的skip是同分辨率恒等相加，U-Net的skip是编码器到解码器的跨分辨率特征拼接，用于恢复空间细节。','两者分别解决什么问题？','ResNet解决梯度传播，U-Net解决下采样丢失的空间细节。'),
    QA('architectures','bonus','mid','为什么卷积核大小和感受野/参数量之间存在权衡？','大核参数多但一层就能获得大感受野；小核参数少但需要堆叠更多层才能达到同样感受野。','实践中怎么选？','看任务对局部细节vs全局上下文的相对需求，以及部署侧的算力预算。'),
    QA('architectures','boundary','low','有效感受野的完整理论解释是否成熟？','有实证测量方法（梯度回传法），但"为什么中心权重远高于边缘"背后的完整理论解释仍在发展中。','怎么在工程里应用这个认知？','设计架构时不能只看理论感受野公式，要实测有效感受野再决策。'),
    QA('architectures','bonus','mid','为什么Transformer检测器的可变形注意力对小目标友好？','稀疏采样可以聚焦到目标附近的若干关键点，而不必对全图做稠密计算，降低了大尺度全局attention对小目标信息的稀释。','和普通self-attention的计算量对比？','可变形注意力是O(n×K)而非O(n²)，K是固定采样点数。'),
    QA('architectures','must','mid','为什么ReLU之后经常直接接归一化层而不是相反顺序？','工程惯例中Conv→Norm→ReLU更常见，让归一化直接作用于卷积输出的原始分布，激活函数在归一化之后引入非线性。','有没有例外？','部分架构尝试Norm在激活之后（Post-Activation变体），效果因架构而异。'),
    QA('architectures','must','mid','为什么检测器的neck(FPN/PAN/BiFPN)如此重要？','单一尺度特征图无法同时兼顾大小目标的检测需求，多尺度特征融合直接决定尺度鲁棒性。','FPN和PAN的区别？','FPN只有自顶向下路径，PAN额外增加自底向上路径补充定位信息。'),
    QA('architectures','bonus','mid','为什么ViT需要patch embedding？','把图像切分成固定大小的patch并线性投影成token序列，让图像能直接输入原本为序列设计的Transformer结构。','patch大小怎么选？','越小patch数越多、计算量越大，但空间分辨率保留得更细。'),
    QA('architectures','bonus','low','SE(Squeeze-and-Excitation)模块的作用是什么？','显式建模通道间依赖关系，用全局信息重新加权各通道特征，是一种轻量级通道注意力。','和普通attention的区别？','SE只在通道维度做注意力，不涉及空间位置的attention计算。'),
]

assert len(ARCHITECTURES) >= 28, len(ARCHITECTURES)
print(f'architectures 题库：{len(ARCHITECTURES)} 题')

In [ ]:
EVAL_STATS = [
    QA('eval_stats','must','high','Accuracy什么时候是坏指标？','类别严重不平衡时，全猜多数类也能有很高accuracy，掩盖稀有类完全学不到的事实。','该看什么？','稀有类的precision/recall，或直接看PR-AUC。'),
    QA('eval_stats','must','high','Precision和Recall分别回答什么问题？','Precision回答"我说是正的里有多少真是正的"，Recall回答"真正的正例我抓到了多少"。','两者的权衡关系是什么？','阈值降低recall通常升高但precision通常降低，反之亦然。'),
    QA('eval_stats','must','mid','F1为什么用调和平均？','调和平均对两者失衡惩罚更重，precision或recall任一极低都会被拉低总分，防止被单边高分数骗过去。','Fβ的β怎么选？','β>1更看重召回，β<1更看重精确率，选择依据是代价矩阵。'),
    QA('eval_stats','must','high','ROC-AUC的概率解释是什么？','等于一个随机正例分数高于一个随机负例分数的概率。','这和排序有什么关系？','它本质是一个排序统计量，衡量排序能力而非绝对错误率。'),
    QA('eval_stats','must','high','不平衡数据下ROC-AUC为什么会骗人？','假阳性率分母里有海量真阴性，稀释了假阳性绝对数量的影响。','该看什么替代？','PR-AUC，它直接受假阳性绝对数量影响。'),
    QA('eval_stats','bonus','mid','PR-AUC为什么对不平衡敏感？','precision的分母是TP+FP，正类越稀少同样数量的FP对precision冲击越大。','和ROC-AUC排序结论会不会矛盾？','不会，两个空间里"谁更好"的排序是一致的，只是数值的可读性不同。'),
    QA('eval_stats','bonus','mid','mAP是怎么算出来的？','对每个类别算PR曲线下面积，检测里还要在多个IoU阈值上再平均。','mAP的局限性有哪些？','平均掩盖表现极差的类，和距离/场景无关，细节见C61-02/C55-05。'),
    QA('eval_stats','must','high','阈值0.5什么时候是错的默认选择？','类别不平衡或两种错误代价不对等时，0.5缺乏依据。','该怎么选阈值？','从代价矩阵算期望代价最小的工作点。'),
    QA('eval_stats','bonus','mid','代价敏感阈值选择的目标函数是什么？','期望代价 = C_FP×FP(阈值) + C_FN×FN(阈值)，选使其最小的阈值。','没有明确代价矩阵怎么办？','退而求其次用Youden J统计量或F1最大点。'),
    QA('eval_stats','must','high','可靠性图怎么画？','按置信度分桶，画每桶平均置信度vs桶内实际准确率，完美校准应落在对角线上。','和ECE的关系？','ECE是可靠性图偏离对角线程度的加权标量总结。'),
    QA('eval_stats','must','high','为什么"高置信不等于高准确"？','现代深度网络普遍过度自信，置信度系统性高于真实正确率。','怎么修？','温度缩放，不改变预测排序只压平置信度。'),
    QA('eval_stats','bonus','mid','温度缩放为什么不改变accuracy？','只是把logit统一除以标量T再过softmax，不改变各类别logit的相对大小顺序。','那它改变了什么？','改变了输出概率的"尖锐程度"，即置信度数值本身。'),
    QA('eval_stats','must','high','正态置信区间什么时候失效？','小样本或比例接近0/1时，覆盖率会显著低于名义值，甚至给出越界的区间。','该用什么替代？','Wilson区间（专为二项比例设计）或bootstrap（通用）。'),
    QA('eval_stats','bonus','mid','Wilson区间相比正态CI的优势？','通过反转显著性检验推导，天然落在[0,1]内，小样本下覆盖率更接近名义值。','大样本下两者有区别吗？','基本重合。'),
    QA('eval_stats','bonus','mid','bootstrap CI的原理和代价？','对样本有放回重采样多次，用重采样统计量的分布估计置信区间；代价是计算量大、对极小样本仍不稳定。','什么场景必须用bootstrap？','没有解析方差公式的统计量，比如中位数、AUC、mAP。'),
    QA('eval_stats','must','high','配对检验和非配对检验怎么选？','同一批测试样本上比较两个模型用配对检验（分数天然相关），不同批样本用非配对检验。','用错了会怎样？','把配对数据当非配对处理，方差被高估，本该显著的结果被判不显著。'),
    QA('eval_stats','must','high','"+0.3 mAP算不算提升"怎么答？','先看多种子方差范围（典型±0.2-0.5），确认用配对还是非配对检验，最后才下结论。','只跑了一个种子怎么办？','诚实说明暂时无法判断是信号还是噪声。'),
    QA('eval_stats','must','high','A/B测试样本量怎么算？','用两比例z检验公式，需要设定显著性水平α和功效power，效应越小样本量按平方反比增长。','效应减半样本量变化多少？','约变为原来的4倍。'),
    QA('eval_stats','bonus','mid','统计功效(power)是什么？','真的有效应时，实验能检测出来的概率，通常要求≥0.8。','功效不够会怎样？','"无显著差异"的结论可能只是样本不够，不是真的没有效应。'),
    QA('eval_stats','bonus','mid','新奇效应(novelty effect)是什么？','用户因为"新"而短期多点击/停留，效应会随时间衰减。','怎么排除？','实验跑够长（覆盖完整周期），看效应是否随时间递减。'),
    QA('eval_stats','must','high','多重比较问题的本质是什么？','同时检验多个假设时，至少一个假阳性的概率会随假设数量指数上升。','怎么校正？','Bonferroni（除以假设数）或控制FDR（Benjamini-Hochberg）。'),
    QA('eval_stats','bonus','mid','peeking(提前偷看)为什么破坏显著性？','反复在多个时间点检验同一假设，等价于隐式多重比较，实际第一类错误率远高于设定α。','怎么避免？','预先固定样本量/时长，或用序贯检验。'),
    QA('eval_stats','must','high','辛普森悖论的本质机制是什么？','分组内趋势一致，合并后反转，因为各组样本权重在方案间分布不同，合并比例被权重主导。','怎么在数据里发现它？','任何整体指标都按关键维度切片复核一遍。'),
    QA('eval_stats','bonus','mid','幸存者偏差怎么在数据里发现？','检查数据收集流程本身是否存在过滤，只观察到"活下来/成功上报"的样本。','有什么典型例子？','线上只收集到成功上报的badcase，真正漏检的往往不在里面。'),
    QA('eval_stats','bonus','mid','回归到均值现象怎么解释？','一次极端表现之后，下一次测量自然更靠近均值，容易被误认为干预生效。','怎么排除这个混淆？','设对照组，只对"上次表现极端"样本做前后对比是触发条件。'),
    QA('eval_stats','must','mid','p-hacking的定义和预防方法？','试了很多切分/指标/模型只报告显著的那个，本质是隐式多重比较；预防要预先注册假设和指标。','和多重比较的关系？','p-hacking是多重比较的一种隐蔽表现形式。'),
    QA('eval_stats','must','high','贝叶斯公式的直觉是什么？','把你已知的P(阳性|病)反过来算你想知道的P(病|阳性)，中间由先验P(病)和总体阳性率归一化。','基率谬误的经典例子？','低患病率+高准确率检测下，阳性结果真阳性概率可能远低于直觉。'),
    QA('eval_stats','must','mid','条件概率和联合概率的区别？','联合概率P(A,B)是两件事同时发生的概率，条件概率P(A|B)是已知B发生后A发生的概率，两者通过P(A,B)=P(A|B)P(B)关联。','什么时候两者相等？','A和B独立时P(A|B)=P(A)。'),
    QA('eval_stats','boundary','low','期望的线性性为什么不需要独立性假设？','E[X+Y]=E[X]+E[Y]对任意随机变量都成立，是积分/求和运算本身的线性性质。','方差呢？','Var(X+Y)在独立（或不相关）时才等于两个方差之和，否则要加协方差项。'),
    QA('eval_stats','bonus','mid','种子方差是什么，为什么会影响"是否算提升"的判断？','同一配置不同随机种子训练结果的自然波动，检测任务典型±0.2-0.5 mAP；不看方差就下结论容易把噪声当信号。','怎么量化需要多少种子？','功效分析(power analysis)，见C61-01。'),
    QA('eval_stats','bonus','low','样本比例不匹配(SRM)是什么？','A/B测试分流本身有bug，两组实际样本比例偏离预期分流比例。','为什么要优先检查SRM？','SRM意味着分流机制本身有问题，此时看任何指标都不可信。'),
    QA('eval_stats','boundary','low','贝叶斯A/B测试和频率派方法的取舍有没有定论？','贝叶斯框架允许持续监控不受多重比较惩罚，但引入先验选择的主观性；工业界两派都在用，没有统一定论。','实践中怎么选？','看团队对先验假设的接受度和持续监控的实际需求。'),
    QA('eval_stats','bonus','mid','Brier Score是什么？','预测概率和真实标签(0/1)之间的均方误差，同时衡量区分度和校准度的综合指标。','和ECE的区别？','Brier Score是连续可微指标，ECE是分桶后的离散近似度量。'),
    QA('eval_stats','bonus','low',"Cohen's Kappa什么时候需要？",'衡量两个标注者一致性时排除随机一致的期望部分，比简单一致率更严格。','检测标注一致性时怎么用？','结合IoU阈值定义"一致"，再计算kappa评估标注质量。'),
    QA('eval_stats','must','mid','为什么报告单一mAP数字而不做显著性检验是危险的？','没有方差参照系，无法判断差异是真实提升还是噪声，容易做出错误的模型选型决策。','正确做法是什么？','多种子训练+配对检验，见C61-01。'),
]

assert len(EVAL_STATS) >= 28, len(EVAL_STATS)
print(f'eval_stats 题库：{len(EVAL_STATS)} 题')

In [ ]:
CV_ENTRY = [
    QA('cv_entry','must','high','IoU的边界情况怎么处理？','两框恰好相切时IoU为0（面积交集为0）；面积为0的退化框需要特殊判断避免除零。','NMS的贪心策略有什么局限？','贪心保留最高分框可能错误抑制相邻的真实目标，见C61-05手撕实现。'),
    QA('cv_entry','must','high','NMS的原理和局限是什么？','按置信度排序，依次保留最高分框并抑制与其IoU超过阈值的其他框；局限是密集/重叠目标场景会错误抑制真实框。','soft-NMS怎么改进？','用连续衰减代替硬性抑制，保留部分置信度而非直接删除。'),
    QA('cv_entry','bonus','mid','标签分配为什么比backbone更影响精度？','分配策略决定了哪些位置被当作正样本训练，直接决定学习信号的质量和数量，见C53-02。','ATSS解决什么问题？','用统计量(均值+标准差)自适应确定IoU阈值，比固定阈值更公平。'),
    QA('cv_entry','bonus','mid','匈牙利匹配解决什么问题？','把预测集合和GT集合的对应关系转化为二分图最优匹配问题，实现一对一分配，见C54-01。','为什么一对一匹配能替代NMS？','训练期就压制重复预测，而不是靠推理期删除重复。'),
    QA('cv_entry','bonus','mid','小目标检测难在哪五个独立原因？','信息量少、IoU对位移极敏感、正样本稀缺、下采样丢失信息、标注误差相对量级大，见C57-01。','NWD相比IoU的优势是什么？','把框建模为高斯分布用Wasserstein距离度量，对尺度不敏感且不相交时仍有梯度。'),
    QA('cv_entry','must','mid','TSR和通用目标检测的五个本质不同是什么？','目标极小、类别极度长尾且区域相关、语义直接影响安全动作、时序信息可用且必须用、误检代价高度不对称，见C55-00。','这对系统设计有什么影响？','需要两级架构、时序融合、代价敏感评测等专门设计。'),
    QA('cv_entry','bonus','mid','两级检测方案vs端到端的误差传播分析？','级联召回=检测召回×分类准确率，这个乘法关系是关键洞察，见C55-02。','两级方案的动机是什么？','长尾解耦、新增类别不重训检测器、分类器可用更高分辨率。'),
    QA('cv_entry','bonus','mid','TIDE式误差分解的六类是什么？','Cls(分类错)/Loc(定位不准)/Both/Dupe(重复检测)/Bkg(背景误检)/Miss(漏检)，见C61-02。','怎么决定下一步该修哪一类？','算出"修好每一类能涨多少mAP"，选收益最大的先修。'),
    QA('cv_entry','must','mid','训练-部署一致性最常见的三个坑是什么？','resize方式不一致(bilinear语义差异)、BGR/RGB顺序颠倒、letterbox实现分歧，见C60-01/04。','怎么定位这类问题？','两边dump张量逐元素diff，二分定位到具体算子。'),
    QA('cv_entry','bonus','mid','长尾类别在检测里的特有解法是什么？','repeat factor sampling（公式r_c=max(1,√(t/f_c))）等重采样策略，见C58-01。','和通用类别不平衡处理有什么不同？','检测里前景-背景不平衡和类别间不平衡要分开处理。'),
    QA('cv_entry','bonus','mid','OHEM和Focal Loss的关系是什么？','OHEM是硬性截断(按loss排序取top-k)，Focal Loss是软性连续加权，见C58-02。','两者哪个更常用？','Focal Loss因为可微、无需额外排序步骤更常见于现代检测器。'),
    QA('cv_entry','bonus','mid','数据闭环的"触发→挖掘→验证"三环节分别做什么？','触发决定回传什么数据，挖掘从海量数据里找出有价值的样本，验证证明加数据真的有用，见C58-00/05。','为什么"加数据"常常无效？','加的是已经会的、分布不对、标注有噪、加完没验证。'),
    QA('cv_entry','must','high','TSR检测输出怎么序列化传给下游的VLA/规控？','结构化字段包括类别、位置、尺寸、置信度、跟踪ID、首次检出时间、关联车道，见C59-03。','置信度为什么必须传递？','不传置信度等于强迫下游把所有检测当真，是感知-决策接口最常见的设计错误。'),
    QA('cv_entry','bonus','mid','RT-DETR为什么能不用NMS？','一对一匹配在训练期就压制了重复预测，加上高效混合编码器和query selection机制，见C53-04。','这带来什么部署优势？','NMS耗时随目标数波动，去掉NMS让端到端延迟更稳定可预测。'),
    QA('cv_entry','bonus','mid','anchor-free和anchor-based的本质区别是什么？','anchor-based预先定义参考框回归偏移量，anchor-free直接预测目标中心/关键点或到边界的距离。','anchor-free的优势是什么？','减少了anchor相关的超参数(尺度/长宽比)，简化了标签分配。'),
    QA('cv_entry','boundary','low','车端多相机(广角+长焦)配置的完整取舍是否有统一最优解？','取决于具体车型的传感器成本预算和检测距离需求，业界没有统一配置标准，见C57-05。','怎么在面试里诚实回答？','说明取舍的关键变量(检测距离/成本/算力)，承认没有放之四海皆准的答案。'),
    QA('cv_entry','bonus','mid','车端部署里"静默回退"是什么现象？','不支持的算子被切成子图交给其他后端，模型能跑但没有变快，见C60-02。','怎么发现这个问题？','看TensorRT分区日志，检查分区数和各分区后端。'),
    QA('cv_entry','bonus','mid','为什么类别数爆炸(限速5-120等变体)需要层次标签设计？','扁平化类别体系会让每个细分类别样本更稀疏，层次标签(粗类→细类)让共享的粗类特征缓解长尾，见C55-01。','层次感知评测怎么设计？','允许粗类正确但细类错误得到部分分数，而不是简单的0/1判定。'),
    QA('cv_entry','must','mid','为什么TSR评测必须按距离/尺寸分桶？','整体mAP会被近处大目标主导，掩盖远处小目标(真正决定安全裕度)的表现，见C55-05/C57-05。','这和C64-04的哪个概念相关？','辛普森悖论——整体指标可能掩盖关键子群体的真实表现。'),
]

assert len(CV_ENTRY) >= 15, len(CV_ENTRY)
print(f'cv_entry 题库：{len(CV_ENTRY)} 题')

In [ ]:
BANK = ML_BASICS + OPTIMIZATION + ARCHITECTURES + EVAL_STATS + CV_ENTRY
ID_TO_TOPIC = {c['id']: c['topic'] for c in BANK}
BY_TOPIC = defaultdict(list)
for c in BANK:
    BY_TOPIC[c['topic']].append(c)

assert len(BANK) >= 150, f'题库应至少 150 题，实际 {len(BANK)}'
assert len(set(c["id"] for c in BANK)) == len(BANK), '题目 id 必须唯一'
for c in BANK:
    assert c['tier'] in ('must', 'bonus', 'boundary'), c
    assert c['freq'] in ('high', 'mid', 'low'), c
    assert all(c[k] for k in ('q', 'a', 'follow', 'pitfall')), c

print(f'题库总量：{len(BANK)} 题，覆盖 {len(BY_TOPIC)} 个主题：')
for topic, cards in BY_TOPIC.items():
    tier_count = defaultdict(int)
    for c in cards:
        tier_count[c['tier']] += 1
    print(f"  {topic:<14} {len(cards):>3} 题  (必答{tier_count['must']} / 加分{tier_count['bonus']} / 边界{tier_count['boundary']})")
print('\n✅ 题库验证通过：150+ 题、字段完整、id 唯一。')

## 3 · SM-2 间隔重复算法

Wozniak 的 SM-2：每次复习后按自评分 `quality`（0-5）更新
**易记因子 EF**（越高说明这张卡越容易记住）、**复习间隔 interval**（天数）、**复习次数 reps**。
`quality < 3` 视为"没记住"，重置复习序列；`quality >= 3` 视为"记住了"，间隔按 `EF` 增长。

In [ ]:
def sm2_update(ef, reps, interval, quality):
    """标准 SM-2 更新公式。quality: 0(完全不会)-5(轻松记住)。"""
    assert 0 <= quality <= 5
    if quality < 3:
        reps = 0
        interval = 1
    else:
        if reps == 0:
            interval = 1
        elif reps == 1:
            interval = 6
        else:
            interval = round(interval * ef)
        reps += 1
    ef = ef + (0.1 - (5 - quality) * (0.08 + (5 - quality) * 0.02))
    ef = max(ef, 1.3)     # EF 有下界，防止"越答错越难记"到负数或过小
    return ef, reps, interval

# 连续 5 次都"轻松记住"(quality=5)：间隔应该越拉越长
ef, reps, interval = 2.5, 0, 0
history = []
for _ in range(5):
    ef, reps, interval = sm2_update(ef, reps, interval, 5)
    history.append((round(ef, 3), reps, interval))

assert history[0] == (2.6, 1, 1)
assert history[1] == (2.7, 2, 6)
assert history[2] == (2.8, 3, 16)
assert history[3] == (2.9, 4, 45)
intervals = [h[2] for h in history]
assert intervals == sorted(intervals), '连续答对时间隔应该单调不减'
print('连续 quality=5：', history)

# 答错(quality=2)应该重置 reps=1、interval=1，且 EF 下降
ef2, reps2, interval2 = sm2_update(2.7, 3, 16, 2)
assert reps2 == 0 and interval2 == 1
assert ef2 < 2.7, 'EF 应该下降'
print(f'\n答对 3 次后突然答错(quality=2)：ef={ef2:.3f}(下降) reps={reps2}(重置) interval={interval2}(重置为1)')
print('✅ SM-2 验证通过：答对间隔越拉越长，答错立刻打回原形并降低 EF。')

## 4 · 闪卡引擎：随机抽题、隐藏答案、按自评分调度

`FlashcardState` 是每张卡片的学习状态；`due_cards` 找出到期的卡片；`pick_card` 从中随机抽一张；
`show_question`/`reveal_answer` 模拟"先隐藏答案，自己想完再翻开"的真实自测流程；`grade` 记录自评分并调度下一次复习。

In [ ]:
def new_state():
    return {'ef': 2.5, 'reps': 0, 'interval': 0, 'due_day': 0, 'total_reviews': 0}

def due_cards(state, today):
    """返回今天到期（due_day <= today）的卡片 id 列表。"""
    return [qid for qid, s in state.items() if s['due_day'] <= today]

def pick_card(state, today, rng):
    """从到期卡片里随机抽一张；没有到期卡片时返回 None。"""
    due = due_cards(state, today)
    if not due:
        return None
    idx = int(rng.integers(0, len(due)))
    return due[idx]

def show_question(card):
    """只暴露题目，模拟"先自己想，再翻答案"。"""
    return card['q']

def reveal_answer(card):
    """翻开答案：一句话答案 / 典型追问 / 踩雷点。"""
    return {'a': card['a'], 'follow': card['follow'], 'pitfall': card['pitfall']}

def grade(state, qid, quality, today):
    """记录自评分，用 SM-2 更新该卡片的调度状态。"""
    s = state[qid]
    ef, reps, interval = sm2_update(s['ef'], s['reps'], s['interval'], quality)
    s['ef'], s['reps'], s['interval'] = ef, reps, interval
    s['due_day'] = today + interval
    s['total_reviews'] += 1

# 冒烟测试：抽一张、看题、翻答案、打分
state0 = {c['id']: new_state() for c in BANK}
rng_demo = np.random.default_rng(1)
qid = pick_card(state0, today=0, rng=rng_demo)
card = next(c for c in BANK if c['id'] == qid)
print('题目：', show_question(card))
ans = reveal_answer(card)
print('一句话答案：', ans['a'])
grade(state0, qid, quality=4, today=0)
assert state0[qid]['total_reviews'] == 1
assert state0[qid]['due_day'] == 1     # 第一次答对(quality>=3, reps 0->1) interval=1
print(f"\n打分后：ef={state0[qid]['ef']:.2f}  下次复习在第 {state0[qid]['due_day']} 天")
print('✅ 闪卡引擎冒烟测试通过：抽题 -> 看题 -> 翻答案 -> 打分 -> 自动调度下一次复习。')

## 5 · 模拟一轮学习：弱主题会被自动"多抽到"

用不同的"模拟掌握概率"给每个主题打分（这里故意把 `optimization` 设成用户的弱项），
跑 60 天、每天最多复习 20 张到期卡片，观察 SM-2 的调度是否真的会让弱主题被复习更多次。

In [ ]:
SIM_SUCCESS_P = {'ml_basics': 0.85, 'architectures': 0.80, 'eval_stats': 0.75,
                  'cv_entry': 0.70, 'optimization': 0.35}   # optimization 是刻意设置的"弱项"

state = {c['id']: new_state() for c in BANK}
rng_sim = np.random.default_rng(11)
DAYS, REVIEWS_PER_DAY = 60, 20

for today in range(DAYS):
    due = due_cards(state, today)
    if not due:
        continue
    k = min(REVIEWS_PER_DAY, len(due))
    idxs = rng_sim.choice(len(due), size=k, replace=False)
    for idx in idxs:
        qid = due[idx]
        topic = ID_TO_TOPIC[qid]
        quality = 5 if rng_sim.random() < SIM_SUCCESS_P[topic] else 2
        grade(state, qid, quality, today)

review_by_topic = defaultdict(int)
for qid, s in state.items():
    review_by_topic[ID_TO_TOPIC[qid]] += s['total_reviews']

print(f"{'主题':<14}{'模拟掌握概率':>10}{'总复习次数':>10}")
for topic in SIM_SUCCESS_P:
    print(f"{topic:<14}{SIM_SUCCESS_P[topic]:>10.2f}{review_by_topic[topic]:>10}")

# 核心断言：掌握概率最低的 optimization，人均复习次数应该明显更高
avg_reviews = {t: review_by_topic[t] / len(BY_TOPIC[t]) for t in SIM_SUCCESS_P}
weakest = min(SIM_SUCCESS_P, key=lambda t: SIM_SUCCESS_P[t])
assert weakest == 'optimization'
assert avg_reviews['optimization'] == max(avg_reviews.values()), avg_reviews
print(f"\n人均复习次数最高的主题：{max(avg_reviews, key=avg_reviews.get)}（应为 optimization）")
print('✅ 验证通过：SM-2 会让"越容易答错"的主题自动获得更多复习机会，不需要手工加权。')

## 6 · 弱项报告生成器

按主题聚合每张卡片的 EF，输出「主题、平均 EF（越低越弱）、卡片数、总复习次数」，按平均 EF **升序**排列（最弱的排最前）。

In [ ]:
def weak_topics_report(state, bank):
    """返回 [(topic, avg_ef, n_cards, total_reviews), ...]，按 avg_ef 升序（最弱在前）。"""
    id_to_topic = {c['id']: c['topic'] for c in bank}
    sums = defaultdict(lambda: [0.0, 0, 0])   # topic -> [ef 累加, 卡片数, 总复习次数]
    for qid, s in state.items():
        t = id_to_topic[qid]
        sums[t][0] += s['ef']
        sums[t][1] += 1
        sums[t][2] += s['total_reviews']
    out = [(t, total_ef / n, n, reviews) for t, (total_ef, n, reviews) in sums.items()]
    return sorted(out, key=lambda row: row[1])

report = weak_topics_report(state, BANK)
print(f"{'主题':<14}{'平均EF':>8}{'卡片数':>8}{'总复习次数':>10}")
for topic, avg_ef, n, reviews in report:
    print(f'{topic:<14}{avg_ef:>8.3f}{n:>8}{reviews:>10}')

assert report[0][0] == 'optimization', '平均 EF 最低（最弱）的应该排在第一位'
assert all(report[i][1] <= report[i+1][1] for i in range(len(report)-1)), '必须按 avg_ef 升序排列'
print('\n✅ 弱项报告验证通过：optimization 排在最前面（最该优先复习）。')

## 7 · 90 分钟复习清单自动排程

按每个主题的弱势程度（`weight = 1/avg_ef`，EF 越低权重越高）分配 90 分钟，
每个主题保证一个最低时长 `floor_min`，**最后一个主题吸收取整误差**，保证总和严格等于 90（同构于 C62-00 的时间预算器）。

In [ ]:
def schedule_90min(report, total_min=90, floor_min=8):
    """report: weak_topics_report() 的输出，已按 avg_ef 升序排列。"""
    topics = [r[0] for r in report]
    avg_ef = [r[1] for r in report]
    n = len(topics)
    weights = [1.0 / e for e in avg_ef]
    wsum = sum(weights)
    remaining = total_min - floor_min * n
    assert remaining >= 0, 'floor_min 太大，总时长不够分'
    out, acc = [], 0
    for i in range(n - 1):
        extra = int(round(remaining * weights[i] / wsum))
        v = floor_min + extra
        out.append((topics[i], v))
        acc += v
    out.append((topics[-1], total_min - acc))    # 最后一项吸收取整误差
    return out

plan = schedule_90min(report)
print(f"{'主题':<14}{'分配分钟数':>10}")
for topic, minutes in plan:
    print(f'{topic:<14}{minutes:>10}')

assert sum(v for _, v in plan) == 90, '总和必须严格等于 90 分钟'
assert plan[0][0] == 'optimization', '最弱的主题应该排在第一位'
assert plan[0][1] == max(v for _, v in plan), '最弱的主题应该分到最多时间'
assert all(v >= 8 for _, v in plan), '每个主题至少保证 8 分钟下限'
print('\n✅ 90 分钟排程验证通过：总和严格等于 90，最弱主题分到最多时间，每主题不低于下限。')

## ✏️ 练习 1：逾期天数

实现 `days_overdue(state_entry, today)`：返回该卡片超过 `due_day` 的天数（还没到期则返回 0）。

In [ ]:
def days_overdue(state_entry, today):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert days_overdue({'due_day': 10}, today=15) == 5
assert days_overdue({'due_day': 20}, today=15) == 0     # 还没到期
assert days_overdue({'due_day': 15}, today=15) == 0      # 恰好今天到期
assert days_overdue({'due_day': 0}, today=30) == 30
print('✅ 练习 1 通过：逾期天数不会是负数，恰好到期算 0 天逾期。')

## ✏️ 练习 2：掌握度分级

实现 `mastery_level(state_entry)`，返回 `'new'` / `'learning'` / `'mastered'`：
- `reps == 0` → `'new'`（还没真正复习过）
- `reps > 0` 且 `interval < 21` → `'learning'`
- `interval >= 21` → `'mastered'`（间隔超过 3 周，视为已掌握）

In [ ]:
def mastery_level(state_entry):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert mastery_level({'reps': 0, 'interval': 0}) == 'new'
assert mastery_level({'reps': 2, 'interval': 6}) == 'learning'
assert mastery_level({'reps': 3, 'interval': 16}) == 'learning'
assert mastery_level({'reps': 4, 'interval': 45}) == 'mastered'
assert mastery_level({'reps': 5, 'interval': 21}) == 'mastered'    # 边界：恰好 21 算 mastered
assert mastery_level({'reps': 5, 'interval': 20}) == 'learning'    # 边界：20 还不算
print('✅ 练习 2 通过：new/learning/mastered 三档边界都正确。')

## ✏️ 练习 3：按主题统计已掌握比例

实现 `mastered_fraction_by_topic(bank, state)`：返回 `{topic: 已掌握比例, ...}`，
"已掌握"用练习 2 的 `mastery_level(...) == 'mastered'` 判定。

In [ ]:
def mastered_fraction_by_topic(bank, state):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 构造一个 3 主题的小型合成状态：X 三张卡里只有 1 张 mastered；Y 全部 mastered；Z 一张都没 mastered
mini_bank = [
    {'id': 'x1', 'topic': 'X'}, {'id': 'x2', 'topic': 'X'}, {'id': 'x3', 'topic': 'X'},
    {'id': 'y1', 'topic': 'Y'}, {'id': 'y2', 'topic': 'Y'},
    {'id': 'z1', 'topic': 'Z'}, {'id': 'z2', 'topic': 'Z'},
]
mini_state = {
    'x1': {'reps': 4, 'interval': 45}, 'x2': {'reps': 1, 'interval': 1}, 'x3': {'reps': 0, 'interval': 0},
    'y1': {'reps': 5, 'interval': 30}, 'y2': {'reps': 6, 'interval': 60},
    'z1': {'reps': 2, 'interval': 6}, 'z2': {'reps': 0, 'interval': 0},
}
frac = mastered_fraction_by_topic(mini_bank, mini_state)
assert abs(frac['X'] - 1/3) < 1e-9, frac['X']
assert abs(frac['Y'] - 1.0) < 1e-9, frac['Y']
assert abs(frac['Z'] - 0.0) < 1e-9, frac['Z']
for t, f in sorted(frac.items()):
    print(f'{t}: 已掌握比例 = {f:.2f}')
print('\n✅ 练习 3 通过：Y 全部掌握、Z 完全没掌握、X 三分之一掌握，与手工构造完全一致。')

## ✏️ 练习 4：诚实「我不知道」的结构检查器

实现 `is_honest_unknown_answer(text)`：检查一段回答是否同时包含 §8 三段式结构的三个部分——
① **承认**（含"我不确定"/"我不知道"/"没有把握"/"not sure"/"don't know"之一）、
② **查证路径**（含"我会查"/"我会验证"/"我会去看"/"查文档"/"verify"/"check"之一）、
③ **相邻已知**（含"相关的是"/"我知道的是"/"但"/"however"/"related to"之一）——三者都满足才返回 `True`。

In [ ]:
def is_honest_unknown_answer(text):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
good = "这个我不确定，我会查文档验证一下原始论文；但相关的是，我知道类似场景下的机制是这样的……"
bad_no_verify = "这个我不知道，但相关的是我记得类似的机制是……"          # 缺查证路径
bad_no_relate = "这个我不确定，我会查文档验证一下。"                      # 缺相邻已知
bad_fabricate = "这个应该是因为学习率太大导致的，反正差不多是这个原因。"   # 缺承认，硬答

assert is_honest_unknown_answer(good) is True
assert is_honest_unknown_answer(bad_no_verify) is False
assert is_honest_unknown_answer(bad_no_relate) is False
assert is_honest_unknown_answer(bad_fabricate) is False

en_good = "I'm not sure about this — I'd check the paper's ablation section to verify; " \
          "however, related to this, I know a similar mechanism works this way in ..."
assert is_honest_unknown_answer(en_good) is True

for label, text in [('good', good), ('缺查证', bad_no_verify), ('缺相邻已知', bad_no_relate),
                     ('硬答(缺承认)', bad_fabricate), ('英文good', en_good)]:
    print(f'{label:<14} -> {is_honest_unknown_answer(text)}')
print('\n✅ 练习 4 通过：三段式结构缺一不可，硬答(fabrication)会被正确识别为不诚实。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def days_overdue(state_entry, today):
    return max(0, today - state_entry['due_day'])

In [ ]:
# 练习 2 参考答案
def mastery_level(state_entry):
    if state_entry['reps'] == 0:
        return 'new'
    if state_entry['interval'] < 21:
        return 'learning'
    return 'mastered'

In [ ]:
# 练习 3 参考答案
def mastered_fraction_by_topic(bank, state):
    id_to_topic = {c['id']: c['topic'] for c in bank}
    by_topic = defaultdict(list)
    for qid, s in state.items():
        by_topic[id_to_topic[qid]].append(mastery_level(s))
    return {t: levels.count('mastered') / len(levels) for t, levels in by_topic.items()}

In [ ]:
# 练习 4 参考答案
def is_honest_unknown_answer(text):
    ack_kw = ['我不确定', '我不知道', '没有把握', 'not sure', "don't know"]
    verify_kw = ['我会查', '我会验证', '我会去看', '我会去查', '查文档', '验证', 'verify', 'check']
    relate_kw = ['相关的是', '我知道的是', '但', 'however', 'related to']
    has_ack = any(k in text for k in ack_kw)
    has_verify = any(k in text for k in verify_kw)
    has_relate = any(k in text for k in relate_kw)
    return has_ack and has_verify and has_relate

---
## 🧪 真实工程胶囊：面试前一周的使用方式 + 90 分钟当天清单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 面试前一周：每天 20 分钟的固定流程
# ══════════════════════════════════════════════════════════════════════
# 1) 跑 §5 的模拟（换成自己的真实状态：用 grade() 记录当天的真实自评分）
# 2) 跑 §6 的 weak_topics_report()，看当前最弱的 1-2 个主题
# 3) 针对最弱主题，回到对应模块（C64-01/02/03/04 或 C18/C53-C61）重读"失效场景"那一段
#    —— 不是重读整节，只读那一段，这才是间隔重复真正节省时间的地方

# ══════════════════════════════════════════════════════════════════════
# B. 面试当天 90 分钟清单（动态版，用 schedule_90min() 的真实输出）
# ══════════════════════════════════════════════════════════════════════
# 1) 跑一遍 weak_topics_report() + schedule_90min()，拿到今天的真实时间表
# 2) 按表执行，最弱主题优先，只做 due 的卡片，不做"看着眼熟就跳过"的判断
# 3) 最后 10 分钟留给 §8 知识边界练习：过一遍"我不确定的领域"清单

# ══════════════════════════════════════════════════════════════════════
# C. 三段式诚实答法（面试中遇到边界层问题时直接套用）
# ══════════════════════════════════════════════════════════════════════
# ① 承认：「这一点我没有把握，不想瞎猜误导你。」
# ② 查证路径：「如果是我，我会先看/去实测/去查……」
# ③ 相邻已知：「但相关的是，我知道……」
# —— 三段都要有，只做①容易显得消极，只做②③容易显得没有正面回应问题

# ══════════════════════════════════════════════════════════════════════
# D. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 白板编码（六步协议 + 数据结构/算法）        -> C62
# · ML 系统设计（七步框架 + 六个案例演练）      -> C63
# · 技术知识问答的详细讲解(本课模块01-04)       -> C64-01/02/03/04
# · 检测专项深挖(IoU/NMS/mAP/匈牙利/标签分配等) -> C18、C53-C61（本模块§6 探针表）
# · 结构化问题求解与沟通(估算/诊断/权衡/模糊需求)-> C65
'''
print(RECIPE)
for token in ['weak_topics_report', 'schedule_90min', '三段式', 'C61', 'C63', 'C65']:
    assert token in RECIPE, token
print('✅ 使用手册覆盖：一周流程 / 当天 90 分钟清单 / 三段式诚实答法 / 课程分工')

### 小结

- **这一模块不引入新知识，它是一台把「已学的东西」转化为「面试当天能用上」的引擎。**
  150+ 题分五大主题（ml_basics/optimization/architectures/eval_stats/cv_entry）内置，
  每题都是「一句话答案 / 典型追问 / 踩雷点」的最小可自测单元。
- **三层分层（必答/加分/边界）对应面试官的真实追问深度**：必答答错是一票否决信号，
  加分是拉开差距的主战场，边界层的正确答案往往是<strong>诚实地说不知道 + 给出查证路径 + 给出相邻已知</strong>。
- **SM-2 间隔重复算法不需要手工判断"我哪里弱"**：只要如实记录每次自评分，
  答错的卡片会自动更频繁地出现，答对的卡片会自动被推迟——弱项报告和 90 分钟排程都建立在这个机制之上。
- **90 分钟复习清单应该是动态的、按真实弱项分配的，而不是平均分配或凭感觉分配**——
  这和 C62-00 的时间预算器是同一个"比例分配 + 最低下限 + 末项吸收误差"模式。
- **知识边界练习不是临场发挥，是提前准备的三段式结构**：承认 + 查证路径 + 相邻已知；
  硬答一个编造的答案，代价往往是连累面试官重新怀疑你之前所有答对的题。
- **本课程到此覆盖了 XPENG 一面三个板块中的全部四门课**（C62 编码 / C63 系统设计 /
  C64 技术知识 / C65 问题求解与沟通）——四门课合起来对应 HR 给出的完整一面形式说明。

面试前最后一次建议：**别再读新内容了，去跑一遍 `weak_topics_report()`。**